In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# QR Code Readability

Modern OCR Pipeline (April 2026)

Primary : PaddleOCR (text detection + recognition, GPU, multilingual).
Extended: PaddleOCR-VL-1.5 (vision-language document parsing).
Timing  : Wall-clock per model stage.
Export  : metrics.json with file-level results + aggregate stats + timing.
Data    : Auto-downloads sample document images at runtime.


## Environment Setup


In [ ]:
import os
# Define __file__ so save-path logic in the pipeline works correctly
__file__ = os.path.abspath('pipeline.py')
%matplotlib inline


## Dataset Setup

Datasets are **auto-downloaded** at runtime via the Kaggle API, sklearn, yfinance, or HuggingFace.

**Kaggle setup** (one-time): place your `kaggle.json` in `~/.kaggle/`.


In [ ]:
# Kaggle API setup — credentials are read from ~/.kaggle/kaggle.json
# To create: https://www.kaggle.com/settings → API → Create New Token
import os, subprocess, sys
try:
    import kaggle  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
# Create project data & output directories
os.makedirs("data", exist_ok=True)
os.makedirs(os.path.join("outputs", "eda"), exist_ok=True)
os.makedirs(os.path.join("outputs", "models"), exist_ok=True)
os.makedirs(os.path.join("outputs", "results"), exist_ok=True)
print("Dataset directories ready")


## Imports & Configuration


In [ ]:
import os, json, time, warnings
from pathlib import Path
import urllib.request

warnings.filterwarnings("ignore")

SAVE_DIR = os.path.dirname(os.path.abspath(__file__))

SAMPLE_URLS = [
    "https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/refs/heads/main/doc/imgs_en/img_12.jpg",
    "https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/refs/heads/main/doc/imgs_en/img623.jpg",
]


## Download Samples


In [ ]:
def download_samples():
    save_dir = Path(SAVE_DIR) / "ocr_samples"
    save_dir.mkdir(exist_ok=True)
    paths = []
    for url in SAMPLE_URLS:
        fname = save_dir / url.split("/")[-1]
        if not fname.exists():
            urllib.request.urlretrieve(url, str(fname))
        paths.append(fname)
    for ext in (".jpg", ".jpeg", ".png", ".bmp", ".tiff"):
        paths.extend([p for p in Path(SAVE_DIR).rglob(f"*{ext}") if p not in paths])
    print(f"{len(paths)} images available for OCR")
    return paths


## Run Paddleocr

PaddleOCR -- primary text detection + recognition.


In [ ]:
def run_paddleocr(files):
    """PaddleOCR -- primary text detection + recognition."""
    from paddleocr import PaddleOCR
    ocr = PaddleOCR(use_angle_cls=True, lang="en", use_gpu=True)
    results = []
    t0 = time.perf_counter()
    for f in files[:30]:
        result = ocr.ocr(str(f), cls=True)
        texts = []
        if result and result[0]:
            for line in result[0]:
                texts.append({"text": line[1][0], "confidence": round(line[1][1], 4)})
        full_text = " ".join(t["text"] for t in texts)
        avg_conf = sum(t["confidence"] for t in texts) / max(len(texts), 1)
        results.append({"file": f.name, "full_text": full_text,
                        "n_lines": len(texts), "avg_confidence": round(avg_conf, 4)})
        preview = full_text[:80] + "..." if len(full_text) > 80 else full_text
        print(f"  {f.name}: {len(texts)} lines (conf {avg_conf:.2f}) -- \'{preview}\'")
    elapsed = time.perf_counter() - t0
    return results, round(elapsed, 1)


## Run Paddleocr Vl

PaddleOCR-VL-1.5 -- vision-language document parsing.


In [ ]:
def run_paddleocr_vl(files):
    """PaddleOCR-VL-1.5 -- vision-language document parsing."""
    from paddleocr import PaddleOCR
    vl_ocr = PaddleOCR(use_doc_orientation_classify=False, use_doc_unwarping=False,
                       use_textline_orientation=False, lang="en", use_gpu=True)
    results = []
    t0 = time.perf_counter()
    for f in files[:10]:
        vl_result = vl_ocr.ocr(str(f), cls=True)
        n_lines = len(vl_result[0]) if vl_result and vl_result[0] else 0
        results.append({"file": f.name, "n_lines": n_lines})
        print(f"  VL-1.5 {f.name}: {n_lines} lines")
    elapsed = time.perf_counter() - t0
    return results, round(elapsed, 1)


## Run Eda

Input file summary for OCR.


In [ ]:
def run_eda(files, save_dir):
    """Input file summary for OCR."""
    print("\n" + "=" * 60)
    print("EXPLORATORY DATA ANALYSIS")
    print("=" * 60)
    print(f"  Input files: {{len(files)}}")
    if files:
        total_size = sum(os.path.getsize(f) for f in files if os.path.isfile(f))
        print(f"  Total size: {{total_size / 1024:.1f}} KB")
    print("EDA complete.")


## Validate Results

Validate OCR outputs for completeness and confidence.


In [ ]:
def validate_results(primary_results, vl_results, save_dir):
    """Validate OCR outputs for completeness and confidence."""
    validation = {
        "paddleocr": {
            "files": len(primary_results),
            "files_with_text": sum(1 for item in primary_results if item.get("n_lines", 0) > 0),
            "avg_confidence": round(
                sum(float(item.get("avg_confidence", 0)) for item in primary_results) / max(len(primary_results), 1),
                4,
            ),
        },
        "paddleocr_vl": {
            "files": len(vl_results),
            "files_with_text": sum(1 for item in vl_results if item.get("n_lines", 0) > 0),
        },
    }
    validation["paddleocr"]["passed"] = validation["paddleocr"]["files_with_text"] > 0
    validation["paddleocr_vl"]["passed"] = validation["paddleocr_vl"]["files_with_text"] > 0 if vl_results else True
    validation["passed"] = validation["paddleocr"]["passed"] or validation["paddleocr_vl"]["passed"]
    out_path = os.path.join(save_dir, "validation.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(validation, f, indent=2)
    print(f"Validation saved to {{out_path}}")
    return validation


## Main Pipeline


In [ ]:
def main():
    print("=" * 60)
    print("OCR | PaddleOCR + PaddleOCR-VL-1.5")
    print("=" * 60)
    files = download_samples()
    run_eda(files, SAVE_DIR)
    metrics = {}
    results = []
    vl_results = []

    # PRIMARY: PaddleOCR
    print()
    print("-- PaddleOCR --")
    try:
        results, elapsed = run_paddleocr(files)
        total_lines = sum(r["n_lines"] for r in results)
        avg_conf = sum(r["avg_confidence"] for r in results) / max(len(results), 1)
        metrics["PaddleOCR"] = {
            "files": len(results), "total_lines": total_lines,
            "avg_confidence": round(avg_conf, 4), "time_s": elapsed,
        }
        print(f"  PaddleOCR: {len(results)} files, {total_lines} lines in {elapsed}s")
    except Exception as e:
        print(f"  PaddleOCR failed: {e}")

    # EXTENDED: PaddleOCR-VL-1.5
    print()
    print("-- PaddleOCR-VL-1.5 (document parsing) --")
    try:
        vl_results, vl_elapsed = run_paddleocr_vl(files)
        vl_total = sum(r["n_lines"] for r in vl_results)
        metrics["PaddleOCR-VL-1.5"] = {
            "files": len(vl_results), "total_lines": vl_total, "time_s": vl_elapsed,
        }
        print(f"  VL-1.5: {len(vl_results)} files, {vl_total} lines in {vl_elapsed}s")
    except Exception as e:
        print(f"  PaddleOCR-VL-1.5 failed: {e}")

    metrics["validation"] = validate_results(results, vl_results, SAVE_DIR)

    # Save metrics
    out_path = os.path.join(SAVE_DIR, "metrics.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, default=str)
    print(f"Metrics saved to {out_path}")

    # Also save detailed per-file results
    detail_path = os.path.join(SAVE_DIR, "ocr_results.json")
    with open(detail_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Detailed results saved to {detail_path}")


## Execute Pipeline

Run the complete ML pipeline end-to-end:


In [ ]:
main()
